# Grid Reliability & Dispatch Optimization

In this notebook, we build an operational optimization layer on top of our electricity demand forecasts.

The objective is to use the predicted nodal loads to determine a feasible generation dispatch while respecting generator capacity, ramping and transmission constraints. The optimization will prioritize serving at least 99.9% of forecast demand while minimizing expensive thermal generation and renewable curtailment.

In [9]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.optimize import linprog

DATA_DIR = Path("../data")

# Load forecasting data
train = pd.read_csv(DATA_DIR / "train.csv")
validation = pd.read_csv(DATA_DIR / "validation.csv")

# Parse datetime
for df in [train, validation]:
    df["datetime"] = pd.to_datetime(
        df["datetime"],
        format="%d-%m-%Y %H:%M"
    )

# Load bonus datasets
generator_limits = pd.read_csv(
    next(DATA_DIR.rglob("generator_operating_limits.csv"))
)

optimization_params = pd.read_csv(
    next(DATA_DIR.rglob("optimization_parameters.csv"))
)

transmission_lines = pd.read_csv(
    next(DATA_DIR.rglob("transmission_lines.csv"))
)

zone_mapping = pd.read_csv(
    next(DATA_DIR.rglob("zone_bus_mapping.csv"))
)

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Generators:", generator_limits.shape)
print("Optimization parameters:", optimization_params.shape)
print("Transmission lines:", transmission_lines.shape)
print("Zone mapping:", zone_mapping.shape)

Train: (34343, 51)
Validation: (7248, 51)
Generators: (12, 15)
Optimization parameters: (6, 5)
Transmission lines: (2, 8)
Zone mapping: (3, 5)


In [10]:
import sys

sys.path.append(str(Path.cwd().parent))

from src.features import (
    build_features,
    get_model_features,
    TARGETS
)

train_fe = build_features(train)
validation_fe = build_features(validation)

model_features = get_model_features(train_fe)

print("Model features:", len(model_features))

Model features: 110


In [11]:
window_counts = validation_fe.groupby("window_id").size()

complete_windows = (
    window_counts[window_counts == 168]
    .index
)

val_complete = validation_fe[
    validation_fe["window_id"].isin(complete_windows)
].copy()

print("Complete windows:", len(complete_windows))
print("Validation rows:", len(val_complete))

Complete windows: 34
Validation rows: 5712


In [12]:
print("\n=== NOTEBOOK 6 SETUP CHECK ===")
print("Train rows:", len(train_fe))
print("Validation rows:", len(validation_fe))
print("Complete validation windows:", len(complete_windows))
print("Complete validation rows:", len(val_complete))
print("Model features:", len(model_features))
print("Generators:", len(generator_limits))
print("Transmission lines:", len(transmission_lines))


=== NOTEBOOK 6 SETUP CHECK ===
Train rows: 34343
Validation rows: 7248
Complete validation windows: 34
Complete validation rows: 5712
Model features: 110
Generators: 12
Transmission lines: 2


In [13]:
from catboost import CatBoostRegressor

# Pick one complete 168-hour validation window
optimization_window = "VAL_20190101"

val_window = (
    validation_fe[
        validation_fe["window_id"] == optimization_window
    ]
    .sort_values("horizon_hour")
    .copy()
)

# Check horizon
print("Rows:", len(val_window))
print(
    "Horizon:",
    val_window["horizon_hour"].min(),
    "→",
    val_window["horizon_hour"].max()
)

regular_predictions = {}
peak_predictions = {}

for target, params in best_configs.items():

    # Regular CatBoost
    regular_model = CatBoostRegressor(
        **params,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1
    )

    regular_model.fit(
        train_fe[model_features],
        train_fe[target]
    )

    regular_pred = regular_model.predict(
        val_window[model_features]
    )

    # Peak-weighted CatBoost
    threshold = train_fe[target].quantile(0.90)

    sample_weights = np.where(
        train_fe[target] >= threshold,
        2.0,
        1.0
    )

    peak_model = CatBoostRegressor(
        **params,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1
    )

    peak_model.fit(
        train_fe[model_features],
        train_fe[target],
        sample_weight=sample_weights
    )

    peak_pred = peak_model.predict(
        val_window[model_features]
    )

    # 50/50 blend
    val_window[f"{target}_pred"] = (
        0.5 * regular_pred +
        0.5 * peak_pred
    )

print("\nForecasts generated.")

display(
    val_window[
        ["datetime", "horizon_hour"] +
        [f"{t}_pred" for t in TARGETS]
    ].head()
)

Rows: 168
Horizon: 1 → 168

Forecasts generated.


,datetime,horizon_hour,nat_demand_pred,load_tocumen_mwh_pred,load_santiago_mwh_pred,load_david_mwh_pred
0,2019-01-01 00:00:00,1,1004.115257,829.298271,64.100682,118.648908
1,2019-01-01 01:00:00,2,979.306250,809.040000,61.905003,115.607716
2,2019-01-01 02:00:00,3,937.937906,774.707802,59.911511,110.926922
3,2019-01-01 03:00:00,4,921.543179,759.089674,58.753099,108.676720
4,2019-01-01 04:00:00,5,901.656472,745.238272,57.800030,106.183091


In [15]:
optimization_params["parameter"] = (
    optimization_params["parameter"]
    .astype(str)
    .str.strip()
)

params = dict(
    zip(
        optimization_params["parameter"],
        optimization_params["value"]
    )
)

print(params)

{'minimum_load_served_fraction': 0.999, 'maximum_unserved_fraction': 0.001, 'renewable_curtailment_penalty_usd_per_mwh': 1.0, 'thermal_variable_cost_usd_per_mwh': 110.0, 'hydro_variable_cost_usd_per_mwh': 12.0, 'renewable_variable_cost_usd_per_mwh': 0.0}


In [16]:
# ============================================================
# 168-HOUR JOINT DISPATCH OPTIMIZATION
# ============================================================

from scipy.optimize import linprog
from scipy.sparse import lil_matrix
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Basic settings
# ------------------------------------------------------------
zones = ["Tocumen", "Santiago", "David"]
n_hours = 168
n_gens = len(generator_limits)

# Generator lookup
gens = generator_limits.reset_index(drop=True)

gen_index = {
    row["generator_id"]: i
    for i, (_, row) in enumerate(gens.iterrows())
}

# ------------------------------------------------------------
# Variable layout
#
# For every hour:
#   12 generator outputs
#   3 unserved-load variables
#   2 transmission flows
# ------------------------------------------------------------

vars_per_hour = n_gens + 3 + 2
n_vars = n_hours * vars_per_hour

def idx_gen(t, g):
    return t * vars_per_hour + g

def idx_shed(t, z):
    return t * vars_per_hour + n_gens + z

def idx_flow(t, f):
    return t * vars_per_hour + n_gens + 3 + f


# ------------------------------------------------------------
# Objective
# ------------------------------------------------------------

c = np.zeros(n_vars)

for t in range(n_hours):

    for g, gen in gens.iterrows():

        cost = float(gen["variable_cost_usd_per_mwh"])

        # Reward renewable utilization by the curtailment penalty.
        # Since curtailment = availability - generation,
        # minimizing curtailment is equivalent to subtracting
        # the penalty from the generation coefficient.
        if gen["renewable"] == 1:
            cost -= params[
                "renewable_curtailment_penalty_usd_per_mwh"
            ]

        c[idx_gen(t, g)] = cost

    # Unserved-load penalty
    for z in range(3):
        c[idx_shed(t, z)] = gens[
            "unserved_load_penalty_usd_per_mwh"
        ].max()


# ------------------------------------------------------------
# Variable bounds
# ------------------------------------------------------------

bounds = []

for t in range(n_hours):

    hour = val_window.iloc[t]

    # Generator bounds
    for g, gen in gens.iterrows():

        technology = str(gen["technology"]).lower()
        zone = gen["zone"]

        lower = float(gen["min_output_mw"])
        upper = float(gen["capacity_mw"])

        # Renewable generation cannot exceed supplied availability
        if technology in ["wind", "solar"]:

            zone_row = zone_mapping[
                zone_mapping["zone"] == zone
            ].iloc[0]

            if technology == "wind":
                renewable_col = zone_row["wind_column"]
            else:
                renewable_col = zone_row["solar_column"]

            availability = max(
                0.0,
                float(hour[renewable_col])
            )

            upper = min(upper, availability)

            # Renewable minimum output must remain zero
            lower = 0.0

        bounds.append((lower, upper))

    # Load shedding bounds
    for z, zone in enumerate(zones):

        target_col = {
            "Tocumen": "load_tocumen_mwh_pred",
            "Santiago": "load_santiago_mwh_pred",
            "David": "load_david_mwh_pred"
        }[zone]

        forecast_load = float(hour[target_col])

        bounds.append(
            (
                0.0,
                params["maximum_unserved_fraction"]
                * forecast_load
            )
        )

    # Transmission flows
    for _, line in transmission_lines.iterrows():

        limit = float(line["thermal_limit_mw"])

        bounds.append((-limit, limit))

# ------------------------------------------------------------
# Equality constraints
#
# Nodal power balance for every hour
#
# Tocumen:
#   generation + shedding - flow_TS = load
#
# Santiago:
#   generation + shedding + flow_TS - flow_SD = load
#
# David:
#   generation + shedding + flow_SD = load
# ------------------------------------------------------------

n_eq = n_hours * 3

A_eq = lil_matrix((n_eq, n_vars))
b_eq = np.zeros(n_eq)

zone_to_index = {
    "Tocumen": 0,
    "Santiago": 1,
    "David": 2
}

for t in range(n_hours):

    hour = val_window.iloc[t]

    for zone in zones:

        z = zone_to_index[zone]
        row = t * 3 + z

        # Generation in this zone
        for g, gen in gens.iterrows():
            if gen["zone"] == zone:
                A_eq[row, idx_gen(t, g)] = 1.0

        # Load shedding
        A_eq[row, idx_shed(t, z)] = 1.0

        # Transmission flows
        if zone == "Tocumen":
            A_eq[row, idx_flow(t, 0)] = -1.0

        elif zone == "Santiago":
            A_eq[row, idx_flow(t, 0)] = 1.0
            A_eq[row, idx_flow(t, 1)] = -1.0

        elif zone == "David":
            A_eq[row, idx_flow(t, 1)] = 1.0

        target_col = {
            "Tocumen": "load_tocumen_mwh_pred",
            "Santiago": "load_santiago_mwh_pred",
            "David": "load_david_mwh_pred"
        }[zone]

        b_eq[row] = float(hour[target_col])


# ------------------------------------------------------------
# Ramp constraints
#
# For every generator:
#
# - ramp_down <= P(t) - P(t-1) <= ramp_up
#
# We start applying this from hour 2 because the initial
# generator state before the forecast horizon is unknown.
# ------------------------------------------------------------

n_ramp = (n_hours - 1) * n_gens * 2

A_ub = lil_matrix((n_ramp, n_vars))
b_ub = np.zeros(n_ramp)

r = 0

for t in range(1, n_hours):

    for g, gen in gens.iterrows():

        ramp_up = float(gen["ramp_up_mw_per_h"])
        ramp_down = float(gen["ramp_down_mw_per_h"])

        current_idx = idx_gen(t, g)
        previous_idx = idx_gen(t - 1, g)

        # P(t) - P(t-1) <= ramp_up
        A_ub[r, current_idx] = 1
        A_ub[r, previous_idx] = -1
        b_ub[r] = ramp_up
        r += 1

        # P(t-1) - P(t) <= ramp_down
        A_ub[r, previous_idx] = 1
        A_ub[r, current_idx] = -1
        b_ub[r] = ramp_down
        r += 1


# ------------------------------------------------------------
# Solve
# ------------------------------------------------------------

print("Variables:", n_vars)
print("Equality constraints:", n_eq)
print("Inequality constraints:", n_ramp)

result = linprog(
    c=c,
    A_ub=A_ub.tocsr(),
    b_ub=b_ub,
    A_eq=A_eq.tocsr(),
    b_eq=b_eq,
    bounds=bounds,
    method="highs"
)

print("\n==============================")
print("168-HOUR OPTIMIZATION RESULT")
print("==============================")

print("Success:", result.success)
print("Status:", result.message)

if not result.success:
    raise RuntimeError(
        "168-hour optimization was infeasible."
    )


# ------------------------------------------------------------
# Extract results
# ------------------------------------------------------------

solution = result.x

dispatch_rows = []

for t in range(n_hours):

    hour = val_window.iloc[t]

    row = {
        "datetime": hour["datetime"],
        "horizon_hour": hour["horizon_hour"]
    }

    # Generator dispatch
    for g, gen in gens.iterrows():

        row[gen["generator_id"]] = solution[
            idx_gen(t, g)
        ]

    # Load shedding
    row["shed_tocumen"] = solution[
        idx_shed(t, 0)
    ]

    row["shed_santiago"] = solution[
        idx_shed(t, 1)
    ]

    row["shed_david"] = solution[
        idx_shed(t, 2)
    ]

    # Transmission flows
    row["flow_tocumen_santiago"] = solution[
        idx_flow(t, 0)
    ]

    row["flow_santiago_david"] = solution[
        idx_flow(t, 1)
    ]

    dispatch_rows.append(row)

dispatch = pd.DataFrame(dispatch_rows)


# ------------------------------------------------------------
# Summary statistics
# ------------------------------------------------------------

generator_columns = [
    gen["generator_id"]
    for _, gen in gens.iterrows()
]

renewable_generators = gens[
    gens["renewable"] == 1
]["generator_id"].tolist()

thermal_generators = gens[
    gens["technology"].str.lower() == "thermal"
]["generator_id"].tolist()

hydro_generators = gens[
    gens["technology"].str.lower() == "hydro"
]["generator_id"].tolist()


total_generation = dispatch[
    generator_columns
].sum().sum()

total_thermal = dispatch[
    thermal_generators
].sum().sum()

total_hydro = dispatch[
    hydro_generators
].sum().sum()

total_renewable = dispatch[
    renewable_generators
].sum().sum()


# Available renewable energy
total_renewable_available = 0.0

for t in range(n_hours):

    hour = val_window.iloc[t]

    for _, gen in gens[
        gens["renewable"] == 1
    ].iterrows():

        zone_row = zone_mapping[
            zone_mapping["zone"] == gen["zone"]
        ].iloc[0]

        if gen["technology"].lower() == "wind":
            col = zone_row["wind_column"]
        else:
            col = zone_row["solar_column"]

        total_renewable_available += max(
            0.0,
            float(hour[col])
        )

total_curtailment = (
    total_renewable_available
    - total_renewable
)


# Load served
total_forecast_load = (
    val_window["load_tocumen_mwh_pred"].sum()
    + val_window["load_santiago_mwh_pred"].sum()
    + val_window["load_david_mwh_pred"].sum()
)

total_unserved = (
    dispatch["shed_tocumen"].sum()
    + dispatch["shed_santiago"].sum()
    + dispatch["shed_david"].sum()
)

load_served_fraction = (
    1 - total_unserved / total_forecast_load
)


print("\n--- Operational Summary ---")
print(
    "Forecast load:",
    round(total_forecast_load, 2),
    "MWh"
)

print(
    "Total generation:",
    round(total_generation, 2),
    "MWh"
)

print(
    "Thermal generation:",
    round(total_thermal, 2),
    "MWh"
)

print(
    "Hydro generation:",
    round(total_hydro, 2),
    "MWh"
)

print(
    "Renewable generation:",
    round(total_renewable, 2),
    "MWh"
)

print(
    "Renewable curtailment:",
    round(total_curtailment, 2),
    "MWh"
)

print(
    "Unserved load:",
    round(total_unserved, 4),
    "MWh"
)

print(
    "Load served:",
    round(load_served_fraction * 100, 4),
    "%"
)

print(
    "Optimization objective:",
    round(result.fun, 2)
)

Variables: 2856
Equality constraints: 504
Inequality constraints: 4008

168-HOUR OPTIMIZATION RESULT
Success: True
Status: Optimization terminated successfully. (HiGHS Status 7: Optimal)

--- Operational Summary ---
Forecast load: 196485.4 MWh
Total generation: 196485.4 MWh
Thermal generation: 61118.4 MWh
Hydro generation: 120098.98 MWh
Renewable generation: 15268.02 MWh
Renewable curtailment: 0.0 MWh
Unserved load: 0.0 MWh
Load served: 100.0 %
Optimization objective: 8148943.73


In [17]:
# ============================================================
# CONSTRAINT VERIFICATION
# ============================================================

print("=== GENERATOR BOUND CHECK ===")

bound_violations = []

for t in range(n_hours):
    hour = val_window.iloc[t]

    for g, gen in gens.iterrows():

        output = dispatch.loc[t, gen["generator_id"]]

        technology = str(gen["technology"]).lower()

        lower = float(gen["min_output_mw"])
        upper = float(gen["capacity_mw"])

        # Renewable availability limit
        if technology in ["wind", "solar"]:

            zone_row = zone_mapping[
                zone_mapping["zone"] == gen["zone"]
            ].iloc[0]

            col = (
                zone_row["wind_column"]
                if technology == "wind"
                else zone_row["solar_column"]
            )

            upper = min(
                upper,
                max(0.0, float(hour[col]))
            )

        if output < lower - 1e-6 or output > upper + 1e-6:
            bound_violations.append({
                "hour": t + 1,
                "generator": gen["generator_id"],
                "output": output,
                "lower": lower,
                "upper": upper
            })

print("Generator bound violations:", len(bound_violations))


print("\n=== RAMP CHECK ===")

ramp_violations = []

for t in range(1, n_hours):

    for g, gen in gens.iterrows():

        current = dispatch.loc[t, gen["generator_id"]]
        previous = dispatch.loc[t - 1, gen["generator_id"]]

        change = current - previous

        ramp_up = float(gen["ramp_up_mw_per_h"])
        ramp_down = float(gen["ramp_down_mw_per_h"])

        if change > ramp_up + 1e-6:
            ramp_violations.append(
                (t + 1, gen["generator_id"], "UP", change, ramp_up)
            )

        if -change > ramp_down + 1e-6:
            ramp_violations.append(
                (t + 1, gen["generator_id"], "DOWN", -change, ramp_down)
            )

print("Ramp violations:", len(ramp_violations))


print("\n=== TRANSMISSION CHECK ===")

flow_violations = []

for t in range(n_hours):

    flow_values = [
        dispatch.loc[t, "flow_tocumen_santiago"],
        dispatch.loc[t, "flow_santiago_david"]
    ]

    for f, (_, line) in enumerate(transmission_lines.iterrows()):

        limit = float(line["thermal_limit_mw"])

        if abs(flow_values[f]) > limit + 1e-6:
            flow_violations.append(
                (t + 1, line["line_id"], flow_values[f], limit)
            )

print("Transmission violations:", len(flow_violations))


print("\n=== LOAD-SERVING CHECK ===")

total_unserved = (
    dispatch["shed_tocumen"].sum()
    + dispatch["shed_santiago"].sum()
    + dispatch["shed_david"].sum()
)

served_fraction = (
    1 - total_unserved / total_forecast_load
)

print("Total unserved:", total_unserved)
print("Load served:", served_fraction * 100, "%")


print("\n=== FINAL CHECK ===")

if (
    len(bound_violations) == 0
    and len(ramp_violations) == 0
    and len(flow_violations) == 0
    and served_fraction >= 0.999
):
    print("✅ ALL CONSTRAINTS SATISFIED")
else:
    print("⚠️ CONSTRAINT ISSUE DETECTED")

=== GENERATOR BOUND CHECK ===
Generator bound violations: 0

=== RAMP CHECK ===
Ramp violations: 0

=== TRANSMISSION CHECK ===
Transmission violations: 0

=== LOAD-SERVING CHECK ===
Total unserved: 0.0
Load served: 100.0 %

=== FINAL CHECK ===
✅ ALL CONSTRAINTS SATISFIED


In [18]:
# ============================================================
# GENERATE 168-HOUR FORECASTS FOR ALL COMPLETE VALIDATION
# WINDOWS
# ============================================================

from catboost import CatBoostRegressor
import numpy as np

# Final tuned configurations
best_configs = {
    "nat_demand": {
        "iterations": 800,
        "depth": 6,
        "learning_rate": 0.05,
        "l2_leaf_reg": 3
    },
    "load_tocumen_mwh": {
        "iterations": 800,
        "depth": 6,
        "learning_rate": 0.05,
        "l2_leaf_reg": 3
    },
    "load_santiago_mwh": {
        "iterations": 800,
        "depth": 5,
        "learning_rate": 0.05,
        "l2_leaf_reg": 3
    },
    "load_david_mwh": {
        "iterations": 800,
        "depth": 6,
        "learning_rate": 0.05,
        "l2_leaf_reg": 3
    }
}

# ------------------------------------------------------------
# Get all complete validation rows
# ------------------------------------------------------------

window_counts = validation_fe.groupby("window_id").size()

complete_windows = (
    window_counts[window_counts == 168]
    .index
)

validation_complete_all = validation_fe[
    validation_fe["window_id"].isin(complete_windows)
].copy()

validation_complete_all = (
    validation_complete_all
    .sort_values(["window_id", "horizon_hour"])
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Generate regular + peak-weighted forecasts
# ------------------------------------------------------------

for target, params_cb in best_configs.items():

    print(f"Training {target}...")

    # Regular model
    regular_model = CatBoostRegressor(
        **params_cb,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1
    )

    regular_model.fit(
        train_fe[model_features],
        train_fe[target]
    )

    regular_pred = regular_model.predict(
        validation_complete_all[model_features]
    )

    # Peak-weighted model
    threshold = train_fe[target].quantile(0.90)

    sample_weights = np.where(
        train_fe[target] >= threshold,
        2.0,
        1.0
    )

    peak_model = CatBoostRegressor(
        **params_cb,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1
    )

    peak_model.fit(
        train_fe[model_features],
        train_fe[target],
        sample_weight=sample_weights
    )

    peak_pred = peak_model.predict(
        validation_complete_all[model_features]
    )

    # 50/50 blend
    validation_complete_all[f"{target}_pred"] = (
        0.5 * regular_pred
        + 0.5 * peak_pred
    )

print("\n==============================")
print("ALL FORECASTS GENERATED")
print("==============================")
print("Windows:", validation_complete_all["window_id"].nunique())
print("Rows:", len(validation_complete_all))
print(
    "Horizon:",
    validation_complete_all["horizon_hour"].min(),
    "→",
    validation_complete_all["horizon_hour"].max()
)

display(
    validation_complete_all[
        ["window_id", "datetime", "horizon_hour"] +
        [f"{target}_pred" for target in TARGETS]
    ].head()
)

Training nat_demand...
Training load_tocumen_mwh...
Training load_santiago_mwh...
Training load_david_mwh...

ALL FORECASTS GENERATED
Windows: 34
Rows: 5712
Horizon: 1 → 168


,window_id,datetime,horizon_hour,nat_demand_pred,load_tocumen_mwh_pred,load_santiago_mwh_pred,load_david_mwh_pred
0,VAL_20190101,2019-01-01 00:00:00,1,1004.115257,829.298271,64.100682,118.648908
1,VAL_20190101,2019-01-01 01:00:00,2,979.306250,809.040000,61.905003,115.607716
2,VAL_20190101,2019-01-01 02:00:00,3,937.937906,774.707802,59.911511,110.926922
3,VAL_20190101,2019-01-01 03:00:00,4,921.543179,759.089674,58.753099,108.676720
4,VAL_20190101,2019-01-01 04:00:00,5,901.656472,745.238272,57.800030,106.183091


In [19]:
# ============================================================
# RUN 168-HOUR OPTIMIZATION FOR ALL 34 VALIDATION WINDOWS
# ============================================================

def optimize_window(val_window):

    n_hours = len(val_window)
    n_gens = len(gens)

    vars_per_hour = n_gens + 3 + 2
    n_vars = n_hours * vars_per_hour

    def idx_gen(t, g):
        return t * vars_per_hour + g

    def idx_shed(t, z):
        return t * vars_per_hour + n_gens + z

    def idx_flow(t, f):
        return t * vars_per_hour + n_gens + 3 + f

    # --------------------------------------------------------
    # Objective
    # --------------------------------------------------------
    c = np.zeros(n_vars)

    for t in range(n_hours):

        for g, gen in gens.iterrows():

            cost = float(gen["variable_cost_usd_per_mwh"])

            if gen["renewable"] == 1:
                cost -= params[
                    "renewable_curtailment_penalty_usd_per_mwh"
                ]

            c[idx_gen(t, g)] = cost

        for z in range(3):
            c[idx_shed(t, z)] = gens[
                "unserved_load_penalty_usd_per_mwh"
            ].max()

    # --------------------------------------------------------
    # Bounds
    # --------------------------------------------------------
    bounds = []

    for t in range(n_hours):

        hour = val_window.iloc[t]

        for g, gen in gens.iterrows():

            technology = str(gen["technology"]).lower()
            zone = gen["zone"]

            lower = float(gen["min_output_mw"])
            upper = float(gen["capacity_mw"])

            if technology in ["wind", "solar"]:

                zone_row = zone_mapping[
                    zone_mapping["zone"] == zone
                ].iloc[0]

                if technology == "wind":
                    col = zone_row["wind_column"]
                else:
                    col = zone_row["solar_column"]

                availability = max(
                    0.0,
                    float(hour[col])
                )

                upper = min(upper, availability)
                lower = 0.0

            bounds.append((lower, upper))

        # Load shedding
        for z, zone in enumerate(
            ["Tocumen", "Santiago", "David"]
        ):

            target_col = {
                "Tocumen": "load_tocumen_mwh_pred",
                "Santiago": "load_santiago_mwh_pred",
                "David": "load_david_mwh_pred"
            }[zone]

            forecast_load = float(hour[target_col])

            bounds.append(
                (
                    0.0,
                    params["maximum_unserved_fraction"]
                    * forecast_load
                )
            )

        # Transmission limits
        for _, line in transmission_lines.iterrows():

            limit = float(line["thermal_limit_mw"])

            bounds.append((-limit, limit))

    # --------------------------------------------------------
    # Nodal balance
    # --------------------------------------------------------
    A_eq = lil_matrix((n_hours * 3, n_vars))
    b_eq = np.zeros(n_hours * 3)

    zones = ["Tocumen", "Santiago", "David"]

    for t in range(n_hours):

        hour = val_window.iloc[t]

        for z, zone in enumerate(zones):

            row = t * 3 + z

            # Generation
            for g, gen in gens.iterrows():

                if gen["zone"] == zone:
                    A_eq[row, idx_gen(t, g)] = 1.0

            # Load shedding
            A_eq[row, idx_shed(t, z)] = 1.0

            # Transmission
            if zone == "Tocumen":
                A_eq[row, idx_flow(t, 0)] = -1.0

            elif zone == "Santiago":
                A_eq[row, idx_flow(t, 0)] = 1.0
                A_eq[row, idx_flow(t, 1)] = -1.0

            elif zone == "David":
                A_eq[row, idx_flow(t, 1)] = 1.0

            target_col = {
                "Tocumen": "load_tocumen_mwh_pred",
                "Santiago": "load_santiago_mwh_pred",
                "David": "load_david_mwh_pred"
            }[zone]

            b_eq[row] = float(hour[target_col])

    # --------------------------------------------------------
    # Ramp constraints
    # --------------------------------------------------------
    n_ramp = (n_hours - 1) * n_gens * 2

    A_ub = lil_matrix((n_ramp, n_vars))
    b_ub = np.zeros(n_ramp)

    r = 0

    for t in range(1, n_hours):

        for g, gen in gens.iterrows():

            ramp_up = float(gen["ramp_up_mw_per_h"])
            ramp_down = float(gen["ramp_down_mw_per_h"])

            current = idx_gen(t, g)
            previous = idx_gen(t - 1, g)

            # Ramp up
            A_ub[r, current] = 1
            A_ub[r, previous] = -1
            b_ub[r] = ramp_up
            r += 1

            # Ramp down
            A_ub[r, previous] = 1
            A_ub[r, current] = -1
            b_ub[r] = ramp_down
            r += 1

    # --------------------------------------------------------
    # Solve
    # --------------------------------------------------------
    result = linprog(
        c=c,
        A_ub=A_ub.tocsr(),
        b_ub=b_ub,
        A_eq=A_eq.tocsr(),
        b_eq=b_eq,
        bounds=bounds,
        method="highs"
    )

    if not result.success:
        return {
            "success": False,
            "message": result.message
        }

    solution = result.x

    # --------------------------------------------------------
    # Calculate summary
    # --------------------------------------------------------
    generator_columns = [
        gen["generator_id"]
        for _, gen in gens.iterrows()
    ]

    thermal_generators = gens[
        gens["technology"].str.lower() == "thermal"
    ]["generator_id"].tolist()

    hydro_generators = gens[
        gens["technology"].str.lower() == "hydro"
    ]["generator_id"].tolist()

    renewable_generators = gens[
        gens["renewable"] == 1
    ]["generator_id"].tolist()

    total_thermal = 0
    total_hydro = 0
    total_renewable = 0

    for t in range(n_hours):

        for g, gen in gens.iterrows():

            output = solution[idx_gen(t, g)]

            if gen["technology"].lower() == "thermal":
                total_thermal += output

            elif gen["technology"].lower() == "hydro":
                total_hydro += output

            elif gen["renewable"] == 1:
                total_renewable += output

    # Available renewable
    total_renewable_available = 0

    for t in range(n_hours):

        hour = val_window.iloc[t]

        for _, gen in gens[
            gens["renewable"] == 1
        ].iterrows():

            zone_row = zone_mapping[
                zone_mapping["zone"] == gen["zone"]
            ].iloc[0]

            if gen["technology"].lower() == "wind":
                col = zone_row["wind_column"]
            else:
                col = zone_row["solar_column"]

            total_renewable_available += max(
                0,
                float(hour[col])
            )

    total_curtailment = (
        total_renewable_available
        - total_renewable
    )

    # Forecast demand
    total_forecast_load = (
        val_window["load_tocumen_mwh_pred"].sum()
        + val_window["load_santiago_mwh_pred"].sum()
        + val_window["load_david_mwh_pred"].sum()
    )

    total_unserved = 0

    for z in range(3):
        total_unserved += solution[
            n_gens + z
            :: vars_per_hour
        ].sum()

    served_fraction = (
        1 - total_unserved / total_forecast_load
    )

    return {
        "success": True,
        "forecast_load_mwh": total_forecast_load,
        "thermal_mwh": total_thermal,
        "hydro_mwh": total_hydro,
        "renewable_mwh": total_renewable,
        "renewable_curtailment_mwh": total_curtailment,
        "unserved_mwh": total_unserved,
        "load_served_pct": served_fraction * 100,
        "objective": result.fun
    }


# ============================================================
# RUN ALL COMPLETE WINDOWS
# ============================================================

optimization_results = []

for window_id in complete_windows:

    print("Optimizing:", window_id)

    window = validation_complete_all[
        validation_complete_all["window_id"] == window_id
    ].sort_values("horizon_hour").copy()

    result = optimize_window(window)

    result["window_id"] = window_id

    optimization_results.append(result)


optimization_results = pd.DataFrame(
    optimization_results
)

print("\n==============================")
print("ALL WINDOWS COMPLETE")
print("==============================")

display(
    optimization_results.round(4)
)

Optimizing: VAL_20190101
Optimizing: VAL_20190108
Optimizing: VAL_20190115
Optimizing: VAL_20190122
Optimizing: VAL_20190129
Optimizing: VAL_20190205
Optimizing: VAL_20190212
Optimizing: VAL_20190219
Optimizing: VAL_20190226
Optimizing: VAL_20190305
Optimizing: VAL_20190312
Optimizing: VAL_20190319
Optimizing: VAL_20190326
Optimizing: VAL_20190402
Optimizing: VAL_20190423
Optimizing: VAL_20190430
Optimizing: VAL_20190507
Optimizing: VAL_20190514
Optimizing: VAL_20190604
Optimizing: VAL_20190625
Optimizing: VAL_20190702
Optimizing: VAL_20190709
Optimizing: VAL_20190730
Optimizing: VAL_20190806
Optimizing: VAL_20190827
Optimizing: VAL_20190903
Optimizing: VAL_20190924
Optimizing: VAL_20191001
Optimizing: VAL_20191022
Optimizing: VAL_20191112
Optimizing: VAL_20191119
Optimizing: VAL_20191126
Optimizing: VAL_20191203
Optimizing: VAL_20191210

ALL WINDOWS COMPLETE


,success,forecast_load_mwh,thermal_mwh,hydro_mwh,renewable_mwh,renewable_curtailment_mwh,unserved_mwh,load_served_pct,objective,window_id
0,True,196485.3996,61118.4,120098.9795,15268.0201,0.0000,0.0,100.0,8.148944e+06,VAL_20190101
1,True,193593.8298,61118.4,112950.6000,19524.8298,0.0000,0.0,100.0,8.058906e+06,VAL_20190108
2,True,197326.1178,61118.4,123770.4971,12437.2208,0.0000,0.0,100.0,8.195833e+06,VAL_20190115
3,True,199659.6956,61118.4,122414.7350,16126.5606,0.0000,0.0,100.0,8.175874e+06,VAL_20190122
4,True,199346.9602,61118.4,124747.5188,13481.0415,0.0000,0.0,100.0,8.206513e+06,VAL_20190129
5,True,199846.9311,61118.4,125174.5957,13553.9354,0.0000,0.0,100.0,8.211565e+06,VAL_20190205
6,True,202208.1196,61118.4,116323.8231,24765.8965,0.0000,0.0,100.0,8.094144e+06,VAL_20190212
7,True,205503.0616,61118.4,126747.8523,17636.8094,0.0000,0.0,100.0,8.226361e+06,VAL_20190219
8,True,194655.4974,61118.4,120386.9668,13150.1306,0.0000,0.0,100.0,8.154517e+06,VAL_20190226
9,True,199373.0699,61118.4,124128.6014,14126.0685,0.0000,0.0,100.0,8.198441e+06,VAL_20190305


In [20]:
from scipy.optimize import milp

print("MILP available:", milp is not None)

MILP available: True


In [21]:
# ============================================================
# 168-HOUR MIXED-INTEGER DISPATCH OPTIMIZATION
# ============================================================

from scipy.optimize import milp, LinearConstraint, Bounds
from scipy.sparse import lil_matrix
import numpy as np
import pandas as pd


def optimize_window_milp(val_window):

    val_window = (
        val_window
        .sort_values("horizon_hour")
        .reset_index(drop=True)
    )

    n_hours = len(val_window)
    n_gens = len(gens)

    zones = ["Tocumen", "Santiago", "David"]

    # --------------------------------------------------------
    # Variable layout per hour
    #
    # Generator outputs      : 12
    # Load shedding           : 3
    # Transmission flows      : 2
    # Thermal commitment      : 3
    # Thermal startup          : 3
    #
    # Total = 23 variables/hour
    # --------------------------------------------------------

    thermal_indices = [
        i for i, row in gens.iterrows()
        if str(row["technology"]).lower() == "thermal"
    ]

    n_thermal = len(thermal_indices)

    vars_per_hour = n_gens + 3 + 2 + n_thermal + n_thermal
    n_vars = n_hours * vars_per_hour

    def idx_gen(t, g):
        return t * vars_per_hour + g

    def idx_shed(t, z):
        return t * vars_per_hour + n_gens + z

    def idx_flow(t, f):
        return t * vars_per_hour + n_gens + 3 + f

    def idx_commit(t, k):
        return t * vars_per_hour + n_gens + 3 + 2 + k

    def idx_startup(t, k):
        return t * vars_per_hour + n_gens + 3 + 2 + n_thermal + k


    # --------------------------------------------------------
    # OBJECTIVE
    # --------------------------------------------------------

    c = np.zeros(n_vars)

    curtailment_penalty = params[
        "renewable_curtailment_penalty_usd_per_mwh"
    ]

    for t in range(n_hours):

        for g, gen in gens.iterrows():

            cost = float(gen["variable_cost_usd_per_mwh"])

            # Renewable curtailment:
            # availability is constant, so minimizing
            # curtailment is equivalent to rewarding generation.
            if gen["renewable"] == 1:
                cost -= curtailment_penalty

            c[idx_gen(t, g)] = cost

        # Unserved-load penalty
        for z in range(3):
            c[idx_shed(t, z)] = gens[
                "unserved_load_penalty_usd_per_mwh"
            ].max()

        # Startup costs
        for k, g in enumerate(thermal_indices):
            c[idx_startup(t, k)] = float(
                gens.loc[g, "startup_cost_usd"]
            )


    # --------------------------------------------------------
    # VARIABLE BOUNDS
    # --------------------------------------------------------

    lower_bounds = np.full(n_vars, -np.inf)
    upper_bounds = np.full(n_vars, np.inf)

    # Generator output bounds
    for t in range(n_hours):

        hour = val_window.iloc[t]

        for g, gen in gens.iterrows():

            technology = str(gen["technology"]).lower()
            zone = gen["zone"]

            lower = 0.0
            upper = float(gen["capacity_mw"])

            # Renewable availability
            if technology in ["wind", "solar"]:

                zone_row = zone_mapping[
                    zone_mapping["zone"] == zone
                ].iloc[0]

                if technology == "wind":
                    renewable_col = zone_row["wind_column"]
                else:
                    renewable_col = zone_row["solar_column"]

                availability = max(
                    0.0,
                    float(hour[renewable_col])
                )

                upper = min(
                    upper,
                    availability
                )

            lower_bounds[idx_gen(t, g)] = lower
            upper_bounds[idx_gen(t, g)] = upper

        # Load shedding
        for z, zone in enumerate(zones):

            target_col = {
                "Tocumen": "load_tocumen_mwh_pred",
                "Santiago": "load_santiago_mwh_pred",
                "David": "load_david_mwh_pred"
            }[zone]

            forecast_load = float(
                hour[target_col]
            )

            lower_bounds[idx_shed(t, z)] = 0.0
            upper_bounds[idx_shed(t, z)] = (
                params["maximum_unserved_fraction"]
                * forecast_load
            )

        # Transmission flows
        for f, (_, line) in enumerate(
            transmission_lines.iterrows()
        ):

            limit = float(line["thermal_limit_mw"])

            lower_bounds[idx_flow(t, f)] = -limit
            upper_bounds[idx_flow(t, f)] = limit

        # Thermal commitment binaries
        for k in range(n_thermal):
            lower_bounds[idx_commit(t, k)] = 0.0
            upper_bounds[idx_commit(t, k)] = 1.0

            lower_bounds[idx_startup(t, k)] = 0.0
            upper_bounds[idx_startup(t, k)] = 1.0


    # --------------------------------------------------------
    # CONSTRAINT MATRIX
    # --------------------------------------------------------

    # Estimate maximum number of constraints
    n_balance = n_hours * 3

    n_commit = n_hours * n_thermal * 2

    n_startup = n_hours * n_thermal * 2

    n_ramp = (n_hours - 1) * n_gens * 2

    total_constraints = (
        n_balance
        + n_commit
        + n_startup
        + n_ramp
    )

    A = lil_matrix(
        (total_constraints, n_vars)
    )

    lb = np.full(
        total_constraints,
        -np.inf
    )

    ub = np.full(
        total_constraints,
        np.inf
    )

    row_id = 0


    # --------------------------------------------------------
    # 1. NODAL POWER BALANCE
    # --------------------------------------------------------

    for t in range(n_hours):

        hour = val_window.iloc[t]

        for z, zone in enumerate(zones):

            target_col = {
                "Tocumen": "load_tocumen_mwh_pred",
                "Santiago": "load_santiago_mwh_pred",
                "David": "load_david_mwh_pred"
            }[zone]

            # Generation in zone
            for g, gen in gens.iterrows():

                if gen["zone"] == zone:
                    A[row_id, idx_gen(t, g)] = 1.0

            # Load shedding
            A[row_id, idx_shed(t, z)] = 1.0

            # Transmission flow
            if zone == "Tocumen":
                A[row_id, idx_flow(t, 0)] = -1.0

            elif zone == "Santiago":
                A[row_id, idx_flow(t, 0)] = 1.0
                A[row_id, idx_flow(t, 1)] = -1.0

            elif zone == "David":
                A[row_id, idx_flow(t, 1)] = 1.0

            load = float(hour[target_col])

            lb[row_id] = load
            ub[row_id] = load

            row_id += 1


    # --------------------------------------------------------
    # 2. THERMAL COMMITMENT
    #
    # G <= capacity * ON
    # G >= minimum_output * ON
    # --------------------------------------------------------

    for t in range(n_hours):

        for k, g in enumerate(thermal_indices):

            gen = gens.loc[g]

            capacity = float(gen["capacity_mw"])
            minimum = float(gen["min_output_mw"])

            # G - capacity * ON <= 0
            A[row_id, idx_gen(t, g)] = 1.0
            A[row_id, idx_commit(t, k)] = -capacity

            ub[row_id] = 0.0

            row_id += 1

            # -G + minimum * ON <= 0
            A[row_id, idx_gen(t, g)] = -1.0
            A[row_id, idx_commit(t, k)] = minimum

            ub[row_id] = 0.0

            row_id += 1


    # --------------------------------------------------------
    # 3. STARTUP LOGIC
    #
    # Startup >= ON(t) - ON(t-1)
    # --------------------------------------------------------

    for t in range(n_hours):

        for k in range(n_thermal):

            current = idx_commit(t, k)
            startup = idx_startup(t, k)

            # ON(t) - STARTUP(t) <= ON(t-1)
            A[row_id, current] = 1.0
            A[row_id, startup] = -1.0

            if t == 0:

                # Initial state assumed OFF
                ub[row_id] = 0.0

            else:

                previous = idx_commit(t - 1, k)
                A[row_id, previous] = -1.0
                ub[row_id] = 0.0

            row_id += 1

            # STARTUP <= ON
            A[row_id, startup] = 1.0
            A[row_id, current] = -1.0

            ub[row_id] = 0.0

            row_id += 1


    # --------------------------------------------------------
    # 4. RAMP CONSTRAINTS
    # --------------------------------------------------------

    for t in range(1, n_hours):

        for g, gen in gens.iterrows():

            ramp_up = float(
                gen["ramp_up_mw_per_h"]
            )

            ramp_down = float(
                gen["ramp_down_mw_per_h"]
            )

            current = idx_gen(t, g)
            previous = idx_gen(t - 1, g)

            # P(t) - P(t-1) <= ramp_up
            A[row_id, current] = 1.0
            A[row_id, previous] = -1.0

            ub[row_id] = ramp_up

            row_id += 1

            # P(t-1) - P(t) <= ramp_down
            A[row_id, previous] = 1.0
            A[row_id, current] = -1.0

            ub[row_id] = ramp_down

            row_id += 1


    # --------------------------------------------------------
    # INTEGER VARIABLES
    # --------------------------------------------------------

    integrality = np.zeros(n_vars)

    for t in range(n_hours):

        for k in range(n_thermal):

            integrality[idx_commit(t, k)] = 1
            integrality[idx_startup(t, k)] = 1


    # --------------------------------------------------------
    # SOLVE MILP
    # --------------------------------------------------------

    constraint = LinearConstraint(
        A.tocsr(),
        lb,
        ub
    )

    result = milp(
        c=c,
        integrality=integrality,
        bounds=Bounds(
            lower_bounds,
            upper_bounds
        ),
        constraints=constraint,
        options={
            "time_limit": 120,
            "mip_rel_gap": 0.001
        }
    )

    # --------------------------------------------------------
    # RESULT
    # --------------------------------------------------------

    if not result.success:
        return {
            "success": False,
            "message": result.message
        }

    solution = result.x

    dispatch_rows = []

    for t in range(n_hours):

        row = {
            "datetime": val_window.iloc[t]["datetime"],
            "horizon_hour": val_window.iloc[t]["horizon_hour"]
        }

        for g, gen in gens.iterrows():

            row[gen["generator_id"]] = (
                solution[idx_gen(t, g)]
            )

        row["shed_tocumen"] = solution[
            idx_shed(t, 0)
        ]

        row["shed_santiago"] = solution[
            idx_shed(t, 1)
        ]

        row["shed_david"] = solution[
            idx_shed(t, 2)
        ]

        row["flow_tocumen_santiago"] = solution[
            idx_flow(t, 0)
        ]

        row["flow_santiago_david"] = solution[
            idx_flow(t, 1)
        ]

        for k, g in enumerate(thermal_indices):

            row[
                f"{gens.loc[g, 'generator_id']}_on"
            ] = solution[idx_commit(t, k)]

            row[
                f"{gens.loc[g, 'generator_id']}_startup"
            ] = solution[idx_startup(t, k)]

        dispatch_rows.append(row)

    dispatch = pd.DataFrame(dispatch_rows)

    # --------------------------------------------------------
    # SUMMARY
    # --------------------------------------------------------

    thermal_generators = gens[
        gens["technology"].str.lower() == "thermal"
    ]["generator_id"].tolist()

    hydro_generators = gens[
        gens["technology"].str.lower() == "hydro"
    ]["generator_id"].tolist()

    renewable_generators = gens[
        gens["renewable"] == 1
    ]["generator_id"].tolist()

    total_thermal = dispatch[
        thermal_generators
    ].sum().sum()

    total_hydro = dispatch[
        hydro_generators
    ].sum().sum()

    total_renewable = dispatch[
        renewable_generators
    ].sum().sum()

    # Renewable availability
    total_renewable_available = 0.0

    for t in range(n_hours):

        hour = val_window.iloc[t]

        for _, gen in gens[
            gens["renewable"] == 1
        ].iterrows():

            zone_row = zone_mapping[
                zone_mapping["zone"] == gen["zone"]
            ].iloc[0]

            if str(gen["technology"]).lower() == "wind":
                col = zone_row["wind_column"]
            else:
                col = zone_row["solar_column"]

            total_renewable_available += max(
                0.0,
                float(hour[col])
            )

    total_curtailment = (
        total_renewable_available
        - total_renewable
    )

    total_load = (
        val_window["load_tocumen_mwh_pred"].sum()
        + val_window["load_santiago_mwh_pred"].sum()
        + val_window["load_david_mwh_pred"].sum()
    )

    total_unserved = (
        dispatch["shed_tocumen"].sum()
        + dispatch["shed_santiago"].sum()
        + dispatch["shed_david"].sum()
    )

    served_pct = (
        1 - total_unserved / total_load
    ) * 100

    startup_count = 0

    for k, g in enumerate(thermal_indices):

        startup_col = (
            f"{gens.loc[g, 'generator_id']}_startup"
        )

        startup_count += round(
            dispatch[startup_col].sum()
        )

    return {
        "success": True,
        "dispatch": dispatch,
        "forecast_load_mwh": total_load,
        "thermal_mwh": total_thermal,
        "hydro_mwh": total_hydro,
        "renewable_mwh": total_renewable,
        "renewable_curtailment_mwh": total_curtailment,
        "unserved_mwh": total_unserved,
        "load_served_pct": served_pct,
        "thermal_startups": startup_count,
        "objective": result.fun,
        "solver_message": result.message
    }

In [22]:
# ============================================================
# TEST MILP ON ONE 168-HOUR WINDOW
# ============================================================

milp_test = optimize_window_milp(
    validation_complete_all[
        validation_complete_all["window_id"]
        == "VAL_20190101"
    ]
)

print("Success:", milp_test["success"])
print("Message:", milp_test.get("solver_message"))

if milp_test["success"]:

    print("\n=== MILP SUMMARY ===")
    print(
        "Forecast load:",
        round(milp_test["forecast_load_mwh"], 2)
    )
    print(
        "Thermal generation:",
        round(milp_test["thermal_mwh"], 2)
    )
    print(
        "Hydro generation:",
        round(milp_test["hydro_mwh"], 2)
    )
    print(
        "Renewable generation:",
        round(milp_test["renewable_mwh"], 2)
    )
    print(
        "Renewable curtailment:",
        round(
            milp_test["renewable_curtailment_mwh"], 2
        )
    )
    print(
        "Unserved load:",
        round(milp_test["unserved_mwh"], 4)
    )
    print(
        "Load served:",
        round(milp_test["load_served_pct"], 4),
        "%"
    )
    print(
        "Thermal startups:",
        milp_test["thermal_startups"]
    )
    print(
        "Objective:",
        round(milp_test["objective"], 2)
    )

Success: True
Message: Optimization terminated successfully. (HiGHS Status 7: Optimal)

=== MILP SUMMARY ===
Forecast load: 196485.4
Thermal generation: 0.0
Hydro generation: 181217.38
Renewable generation: 15268.02
Renewable curtailment: 0.0
Unserved load: 0.0
Load served: 100.0 %
Thermal startups: 0
Objective: 2159340.53


In [23]:
milp_results = []

for window_id in complete_windows:

    print("Optimizing:", window_id)

    window = validation_complete_all[
        validation_complete_all["window_id"] == window_id
    ].copy()

    result = optimize_window_milp(window)

    summary = {
        "window_id": window_id,
        "success": result["success"]
    }

    if result["success"]:
        summary.update({
            "forecast_load_mwh": result["forecast_load_mwh"],
            "thermal_mwh": result["thermal_mwh"],
            "hydro_mwh": result["hydro_mwh"],
            "renewable_mwh": result["renewable_mwh"],
            "renewable_curtailment_mwh": result["renewable_curtailment_mwh"],
            "unserved_mwh": result["unserved_mwh"],
            "load_served_pct": result["load_served_pct"],
            "thermal_startups": result["thermal_startups"],
            "objective": result["objective"]
        })

    milp_results.append(summary)

milp_results = pd.DataFrame(milp_results)

print("\n=== ALL 34 WINDOWS ===")
display(milp_results.round(3))

Optimizing: VAL_20190101
Optimizing: VAL_20190108
Optimizing: VAL_20190115
Optimizing: VAL_20190122
Optimizing: VAL_20190129
Optimizing: VAL_20190205
Optimizing: VAL_20190212
Optimizing: VAL_20190219
Optimizing: VAL_20190226
Optimizing: VAL_20190305
Optimizing: VAL_20190312
Optimizing: VAL_20190319
Optimizing: VAL_20190326
Optimizing: VAL_20190402
Optimizing: VAL_20190423
Optimizing: VAL_20190430
Optimizing: VAL_20190507
Optimizing: VAL_20190514
Optimizing: VAL_20190604
Optimizing: VAL_20190625
Optimizing: VAL_20190702
Optimizing: VAL_20190709
Optimizing: VAL_20190730
Optimizing: VAL_20190806
Optimizing: VAL_20190827
Optimizing: VAL_20190903
Optimizing: VAL_20190924
Optimizing: VAL_20191001
Optimizing: VAL_20191022
Optimizing: VAL_20191112
Optimizing: VAL_20191119
Optimizing: VAL_20191126
Optimizing: VAL_20191203
Optimizing: VAL_20191210

=== ALL 34 WINDOWS ===


,window_id,success,forecast_load_mwh,thermal_mwh,hydro_mwh,renewable_mwh,renewable_curtailment_mwh,unserved_mwh,load_served_pct,thermal_startups,objective
0,VAL_20190101,True,196485.400,0.0,181217.380,15268.020,0.00,0.0,100.0,0,2159340.534
1,VAL_20190108,True,193593.830,0.0,174069.000,19524.830,0.00,0.0,100.0,0,2069303.170
2,VAL_20190115,True,197326.118,-0.0,184888.897,12437.221,-0.00,0.0,100.0,0,2206229.544
3,VAL_20190122,True,199659.696,0.0,183533.135,16126.561,0.00,0.0,100.0,0,2186271.060
4,VAL_20190129,True,199346.960,0.0,185865.919,13481.041,0.00,0.0,100.0,0,2216909.984
5,VAL_20190205,True,199846.931,-0.0,186292.996,13553.935,-0.00,0.0,100.0,0,2221962.013
6,VAL_20190212,True,202208.120,0.0,177442.223,24765.897,0.00,0.0,100.0,0,2104540.780
7,VAL_20190219,True,205503.062,-0.0,187866.252,17636.809,0.00,0.0,100.0,0,2236758.218
8,VAL_20190226,True,194655.497,-0.0,181505.367,13150.131,0.00,0.0,100.0,0,2164914.271
9,VAL_20190305,True,199373.070,-0.0,185247.001,14126.068,-0.00,0.0,100.0,0,2208837.948


In [24]:
print("Successful windows:", milp_results["success"].sum())
print("Failed windows:", (~milp_results["success"]).sum())

print("\nMinimum load served:",
      milp_results["load_served_pct"].min(), "%")

print("Total unserved:",
      milp_results["unserved_mwh"].sum())

print("Total thermal:",
      milp_results["thermal_mwh"].sum())

print("Total renewable curtailment:",
      milp_results["renewable_curtailment_mwh"].sum())

Successful windows: 34
Failed windows: 0

Minimum load served: 100.0 %
Total unserved: 0.0
Total thermal: -1.6653692314072543e-13
Total renewable curtailment: 9.140311129156544


In [25]:
bonus_summary = {
    "Windows": len(milp_results),
    "Feasible_Windows": milp_results["success"].sum(),
    "Min_Load_Served_%": milp_results["load_served_pct"].min(),
    "Total_Unserved_MWh": milp_results["unserved_mwh"].sum(),
    "Total_Thermal_MWh": milp_results["thermal_mwh"].sum(),
    "Total_Hydro_MWh": milp_results["hydro_mwh"].sum(),
    "Total_Renewable_MWh": milp_results["renewable_mwh"].sum(),
    "Total_Curtailment_MWh": milp_results["renewable_curtailment_mwh"].sum(),
    "Total_Thermal_Startups": milp_results["thermal_startups"].sum(),
    "Total_Operating_Cost_USD": milp_results["objective"].sum()
}

display(
    pd.DataFrame(
        bonus_summary.items(),
        columns=["Metric", "Value"]
    )
)

,Metric,Value
0,Windows,3.400000e+01
1,Feasible_Windows,3.400000e+01
2,Min_Load_Served_%,1.000000e+02
3,Total_Unserved_MWh,0.000000e+00
4,Total_Thermal_MWh,-1.665369e-13
5,Total_Hydro_MWh,6.092396e+06
6,Total_Renewable_MWh,9.172054e+05
7,Total_Curtailment_MWh,9.140311e+00
8,Total_Thermal_Startups,0.000000e+00
9,Total_Operating_Cost_USD,7.219154e+07
